# DARKMAP-Q — YOLO Detection Test (images only)

No camera. No hardware. Downloads two public YOLO demo photos, runs YOLO on them,
filters to mission classes, draws bboxes, and shows how the detections are placed
on the 2D recon map.

**Cells degrade gracefully** — synthetic fallback data is used when `ultralytics` is not installed.

In [ ]:
import sys, os, math
import matplotlib
matplotlib.use("Agg")  # remove this line for interactive plots in Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import numpy as np

REPO  = os.path.abspath(os.path.join(os.getcwd(), ".."))
LINUX = os.path.join(REPO, "linux")
sys.path.insert(0, LINUX)
print("Repo:", REPO)
print("Linux path added:", LINUX)

In [ ]:
import importlib

for mod, pip in [("cv2", "opencv-python"), ("ultralytics", "ultralytics"),
                 ("numpy", None), ("matplotlib", None)]:
    try:
        m = importlib.import_module(mod)
        print(f"  ok  {mod:<14} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING  {mod:<14} -> pip install {pip or mod}")

> **Missing `ultralytics`?**  
> `pip install ultralytics`  then restart the kernel.  
> All cells run without it — they use synthetic bounding boxes as a fallback.

## 1 · Category mapping

In [ ]:
from detector import CATEGORY_OF, display_label, MISSION_CLASSES
from mapper   import CATEGORY_COLORS

print(f"{len(MISSION_CLASSES)} mission classes -> 4 recon categories\n")
by_cat = {}
for coco, cat in CATEGORY_OF.items():
    by_cat.setdefault(cat, []).append(display_label(coco))

for cat, labels in by_cat.items():
    col = CATEGORY_COLORS.get(cat, "#ccc")
    print(f"  {cat:<14}  {col}   {', '.join(sorted(labels))}")

## 2 · Download test images

In [ ]:
import urllib.request, urllib.error, ssl, pathlib, cv2

IMG_DIR = pathlib.Path("/tmp/darkmap_test_imgs")
IMG_DIR.mkdir(exist_ok=True)

# Bypass macOS SSL cert issue for public demo images only.
_ssl_ctx = ssl.create_default_context()
_ssl_ctx.check_hostname = False
_ssl_ctx.verify_mode = ssl.CERT_NONE

# Real-world scenes pre-verified to contain diverse mission objects.
# picsum id=1  : person 91% + laptop 92% + phone 71% + chair 34%  (human/electronics/environment)
# picsum id=513: person 90% + chair 75% + dining-table 63%        (human/environment)
# picsum id=800: person 90% + suitcase 85%                        (human/bag)
SOURCES = [
    ("workspace.jpg",   "https://picsum.photos/id/1/800/600"),
    ("restaurant.jpg",  "https://picsum.photos/id/513/800/600"),
    ("traveller.jpg",   "https://picsum.photos/id/800/800/600"),
]

def _download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=_ssl_ctx, timeout=15) as r, \
         open(dest, "wb") as f:
        f.write(r.read())

images = {}  # fname -> numpy BGR array
for fname, url in SOURCES:
    dest = IMG_DIR / fname
    # Always re-download to get the right image (IDs changed)
    try:
        _download(url, str(dest))
        print(f"Downloaded: {fname}")
    except Exception as e:
        print(f"  {fname} failed: {e}")
    if dest.exists():
        img = cv2.imread(str(dest))
        if img is not None:
            images[fname] = img
            print(f"  Loaded {fname}  {img.shape[1]}x{img.shape[0]}")

if not images:
    img = np.full((480, 640, 3), 30, dtype=np.uint8)
    cv2.putText(img, "synthetic (no network)", (80, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (180, 180, 180), 2)
    images["synthetic.jpg"] = img
    print("No network: using synthetic image")

print(f"\nImages ready: {list(images.keys())}")

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(7 * len(images), 5))
if len(images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=11)
    ax.axis("off")
plt.suptitle("Test images (input to YOLO)", fontsize=13)
plt.tight_layout()
OUT = "/tmp/darkmap_input_images.png"
plt.savefig(OUT, dpi=120, bbox_inches="tight")
from IPython.display import display, Image as _Img
display(_Img(filename=OUT))
plt.show()
print(f"Saved -> {OUT}")

## 3 · Run YOLO and filter to mission classes

In [ ]:
YOLO_AVAILABLE = False
model = None
try:
    from ultralytics import YOLO
    model = YOLO("yolov8n.pt")  # ~6 MB, downloads once then cached offline
    YOLO_AVAILABLE = True
    print("Model loaded: yolov8n")
except ImportError:
    print("ultralytics not installed -> using synthetic bounding boxes")
    print("pip install ultralytics  then restart kernel for live inference")
except Exception as e:
    print(f"Model error: {e}")

In [ ]:
# Synthetic bboxes used when ultralytics is not installed.
# These approximate what YOLO actually returns on these two images.
FALLBACK = {
    "zidane.jpg": [
        {"coco": "person",   "conf": 0.92, "bbox": (167, 30,  536, 720), "cx_frac": 0.47},
        {"coco": "person",   "conf": 0.84, "bbox": (460, 65,  640, 690), "cx_frac": 0.86},
        {"coco": "backpack", "conf": 0.56, "bbox": (370, 400, 530, 620), "cx_frac": 0.70},
    ],
    "bus.jpg": [
        {"coco": "person",   "conf": 0.90, "bbox": ( 10, 230,  80, 780), "cx_frac": 0.07},
        {"coco": "person",   "conf": 0.88, "bbox": (185, 200, 260, 740), "cx_frac": 0.34},
        {"coco": "person",   "conf": 0.75, "bbox": (285, 220, 360, 755), "cx_frac": 0.50},
        {"coco": "person",   "conf": 0.68, "bbox": (490, 225, 580, 720), "cx_frac": 0.82},
    ],
    "synthetic.jpg": [],
}

all_results = {}  # fname -> list of mission dets

for name, img in images.items():
    if YOLO_AVAILABLE:
        raw = model(img, conf=0.30, verbose=False)
        dets = []
        w = img.shape[1]
        for res in raw:
            names_map = getattr(res, "names", {})
            for box in (getattr(res, "boxes", None) or []):
                try:
                    cls_id = int(box.cls[0])
                    conf   = float(box.conf[0])
                    x1,y1,x2,y2 = [float(v) for v in box.xyxy[0]]
                    coco   = names_map.get(cls_id, str(cls_id))
                except Exception:
                    continue
                if coco not in CATEGORY_OF:
                    continue
                cx_frac = min(1.0, max(0.0, (x1+x2)/2/w))
                dets.append({"coco": coco, "conf": round(conf,3),
                              "bbox": (x1,y1,x2,y2), "cx_frac": round(cx_frac,3)})
    else:
        dets = FALLBACK.get(name, [])

    all_results[name] = dets
    src = "live YOLO" if YOLO_AVAILABLE else "synthetic fallback"
    print(f"\n{name}  [{src}]  {len(dets)} mission detections")
    for d in dets:
        cat = CATEGORY_OF.get(d["coco"], "?")
        lbl = display_label(d["coco"])
        print(f"  {cat:<14} {lbl:<12} conf={d['conf']:.0%}  cx_frac={d['cx_frac']:.2f}")

## 4 · Detection visualisation

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(8 * len(images), 6))
if len(images) == 1:
    axes = [axes]

for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    seen_cats = set()
    for d in all_results[name]:
        cat = CATEGORY_OF.get(d["coco"], "?")
        lbl = display_label(d["coco"])
        col = CATEGORY_COLORS.get(cat, "#c0c0c0")
        x1, y1, x2, y2 = d["bbox"]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            boxstyle="round,pad=2", linewidth=2.5,
            edgecolor=col, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1+3, max(y1-6, 8), f"{lbl} {d['conf']:.0%}",
                color="white", fontsize=8, fontweight="bold",
                bbox=dict(facecolor=col, alpha=0.85, pad=1.5, edgecolor="none"))
        seen_cats.add(cat)

    handles = [mpatches.Patch(facecolor=CATEGORY_COLORS.get(c, "#ccc"), label=c)
               for c in sorted(seen_cats)]
    if handles:
        ax.legend(handles=handles, loc="lower right", framealpha=0.8, fontsize=8)
    src = "live YOLO" if YOLO_AVAILABLE else "synthetic boxes"
    ax.set_title(f"{name}  [{src}]", fontsize=11)
    ax.axis("off")

plt.suptitle("Mission-class detections", fontsize=14)
plt.tight_layout()
OUT = "/tmp/darkmap_detections.png"
plt.savefig(OUT, dpi=140, bbox_inches="tight")
from IPython.display import display, Image as _Img
display(_Img(filename=OUT))
plt.show()
print(f"Saved -> {OUT}")

## 5 · Mapper tag placement

Simulate the rover driving forward 120 cm, scanning a wall, then receiving
the detections from the first image and placing them on the 2D map.

In [ ]:
from mapper import Mapper

mapper = Mapper()

# Move rover forward
mapper.apply_move("FORWARD", duration_ms=3000)
print(f"Pose: ({mapper.pose.x:.1f}, {mapper.pose.y:.1f})  heading={math.degrees(mapper.pose.theta):.0f} deg")

# Simulated scan sweep: wall ~130 cm straight ahead
for angle in range(-90, 91, 10):
    dist = 130 if abs(angle) < 25 else 260
    mapper.add_scan(angle, dist)
print(f"Obstacle points from scan: {len(mapper.obstacle_points)}")

# Place detection tags
HFOV = 60.0  # degrees, matches Detector default
first_name = list(all_results.keys())[0]
dets = all_results[first_name]
print(f"\nTagging {len(dets)} detections from '{first_name}':")

for i, d in enumerate(dets):
    bearing  = (0.5 - d["cx_frac"]) * HFOV   # left of frame = positive deg
    distance = 90 + i * 8                     # simulated front ultrasonic readings
    cat = CATEGORY_OF.get(d["coco"], "?")
    lbl = display_label(d["coco"])
    tag = mapper.add_detection_tag(
        category    = cat,
        label       = lbl,
        conf        = d["conf"],
        distance_cm = distance,
        bearing_deg = bearing,
    )
    print(f"  {cat:<14} {lbl:<12}  dist={distance}cm  bear={bearing:+.1f}deg  -> ({tag['x']:.1f}, {tag['y']:.1f}) cm")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), facecolor="#060d16")
ax.set_facecolor("#060d16")

if len(mapper.path_points) >= 2:
    xs, ys = zip(*mapper.path_points)
    ax.plot(xs, ys, color="#4ea8ff", linewidth=2, label="path")

if mapper.obstacle_points:
    ox, oy = zip(*mapper.obstacle_points)
    ax.scatter(ox, oy, s=10, color="#ff5470", label="obstacle", zorder=3, alpha=0.6)

seen = set()
for t in mapper.detection_tags:
    col = CATEGORY_COLORS.get(t["category"], "#c0c0c0")
    lbl = t["category"] if t["category"] not in seen else None
    seen.add(t["category"])
    ax.scatter([t["x"]], [t["y"]], s=100, marker="D",
               facecolors="none", edgecolors=col, linewidths=2.2, zorder=6, label=lbl)
    ax.annotate(t["label"], (t["x"], t["y"]),
                textcoords="offset points", xytext=(8, 4),
                fontsize=8, color=col, fontweight="bold", zorder=7)

ax.plot(mapper.pose.x, mapper.pose.y, marker="o", ms=11, color="#22d3a0", zorder=8, label="rover")
ax.arrow(mapper.pose.x, mapper.pose.y,
         20*math.cos(mapper.pose.theta), 20*math.sin(mapper.pose.theta),
         head_width=7, head_length=7, fc="#22d3a0", ec="#22d3a0", zorder=9)

ax.set_aspect("equal", adjustable="datalim")
ax.tick_params(colors="#4a6a88")
for sp in ax.spines.values(): sp.set_color("#1e2d42")
ax.set_xlabel("X (cm)", color="#4a6a88")
ax.set_ylabel("Y (cm)", color="#4a6a88")
src = "live YOLO" if YOLO_AVAILABLE else "synthetic"
ax.set_title(f"2D recon map with detection tags [{src}]", color="#c8d8e8", fontsize=12)
ax.legend(loc="upper left", framealpha=0.6, fontsize=8, facecolor="#0d1824", labelcolor="#c8d8e8")

plt.tight_layout()
OUT = "/tmp/darkmap_map_with_tags.png"
plt.savefig(OUT, dpi=140, bbox_inches="tight")
from IPython.display import display, Image as _Img
display(_Img(filename=OUT))
plt.show()
print(f"Saved -> {OUT}")

## 6 · Points CSV preview

In [ ]:
CSV_PATH = "/tmp/darkmap_points_test.csv"
mapper.save_points_csv(CSV_PATH)

with open(CSV_PATH) as f:
    lines = f.readlines()

det_rows  = [l for l in lines if l.startswith("detection")]
path_rows = [l for l in lines if l.startswith("path")]
obs_rows  = [l for l in lines if l.startswith("obstacle")]

print(f"Header:          {lines[0].strip()}")
print(f"Path rows:       {len(path_rows)}")
print(f"Obstacle rows:   {len(obs_rows)}")
print(f"Detection rows:  {len(det_rows)}")
if det_rows:
    print("\nDetection rows:")
    for r in det_rows:
        print(" ", r.strip())

---
## 7 · Edge quantization — FP32 vs INT8

Export `yolov8n.pt` to ONNX FP32, then dynamic-quantize to INT8 using
`onnxruntime.quantization.quantize_dynamic`.  No calibration dataset needed.
Deploy the INT8 file to the UNO Q — only `onnxruntime` required, no PyTorch.

In [ ]:
import os, sys, time
sys.path.insert(0, LINUX)

FP32_PATH = os.path.join(LINUX, 'models', 'yolov8n.onnx')
INT8_PATH = os.path.join(LINUX, 'models', 'yolov8n_int8.onnx')

# --- Export FP32 (skip if already done) ---
if not os.path.exists(FP32_PATH):
    print('Exporting yolov8n.pt -> ONNX FP32 ...')
    from ultralytics import YOLO
    model_pt = YOLO('yolov8n.pt')
    exported = model_pt.export(format='onnx', imgsz=416, simplify=True, opset=12)
    import shutil
    os.makedirs(os.path.dirname(FP32_PATH), exist_ok=True)
    shutil.move(str(exported), FP32_PATH)
else:
    print(f'FP32 already exists: {FP32_PATH}')

fp32_mb = os.path.getsize(FP32_PATH) / 1e6
print(f'FP32 size: {fp32_mb:.1f} MB')

# --- Quantize to INT8 (skip if already done) ---
if not os.path.exists(INT8_PATH):
    print('Quantizing FP32 -> INT8 (dynamic, no calibration needed) ...')
    from onnxruntime.quantization import quantize_dynamic, QuantType
    t0 = time.time()
    quantize_dynamic(FP32_PATH, INT8_PATH, weight_type=QuantType.QUInt8)
    print(f'Done in {time.time()-t0:.1f}s')
else:
    print(f'INT8 already exists: {INT8_PATH}')

int8_mb = os.path.getsize(INT8_PATH) / 1e6
saving   = fp32_mb - int8_mb
print(f'INT8 size: {int8_mb:.1f} MB  ({saving:.1f} MB saved,  {int8_mb/fp32_mb*100:.0f}% of FP32)')

### Speed benchmark — FP32 vs INT8 on a real image

In [ ]:
import onnxruntime as ort

BENCH_IMG = list(images.values())[0]   # use first test image
RUNS = 20

def _preprocess(img, sz=416):
    blob = cv2.resize(img, (sz, sz))
    blob = blob[:, :, ::-1].astype(np.float32) / 255.0
    return blob.transpose(2, 0, 1)[np.newaxis]

blob = _preprocess(BENCH_IMG)

results_bench = {}
for label, path in [('FP32', FP32_PATH), ('INT8', INT8_PATH)]:
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    inp  = sess.get_inputs()[0].name
    # Warm-up
    for _ in range(3):
        sess.run(None, {inp: blob})
    # Timed runs
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        sess.run(None, {inp: blob})
        times.append((time.perf_counter() - t0) * 1000)
    median_ms = float(np.median(times))
    results_bench[label] = median_ms
    print(f'  {label}  median={median_ms:.1f}ms  min={min(times):.1f}ms  max={max(times):.1f}ms')

speedup = results_bench['FP32'] / results_bench['INT8']
print(f'\nINT8 is {speedup:.2f}x faster than FP32 on this CPU')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), facecolor='#060d16')

for ax in (ax1, ax2):
    ax.set_facecolor('#060d16')
    ax.tick_params(colors='#4a6a88')
    for sp in ax.spines.values(): sp.set_color('#1e2d42')

# Model size
labels = ['FP32', 'INT8']
sizes  = [fp32_mb, int8_mb]
bars1  = ax1.bar(labels, sizes, color=['#4ea8ff', '#22d3a0'], width=0.5)
for bar, val in zip(bars1, sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f} MB', ha='center', color='#c8d8e8', fontsize=11, fontweight='bold')
ax1.set_ylabel('Size (MB)', color='#4a6a88')
ax1.set_title('Model size', color='#c8d8e8')
ax1.set_ylim(0, fp32_mb * 1.25)

# Inference speed
speeds = [results_bench['FP32'], results_bench['INT8']]
bars2  = ax2.bar(labels, speeds, color=['#4ea8ff', '#22d3a0'], width=0.5)
for bar, val in zip(bars2, speeds):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}ms', ha='center', color='#c8d8e8', fontsize=11, fontweight='bold')
ax2.set_ylabel('Median inference (ms)', color='#4a6a88')
ax2.set_title('Inference speed (lower is better)', color='#c8d8e8')
ax2.set_ylim(0, max(speeds) * 1.25)

plt.suptitle('FP32 vs INT8 ONNX on ARM64 (CPUExecutionProvider)', color='#c8d8e8', fontsize=13)
plt.tight_layout()
OUT = '/tmp/darkmap_quant_bench.png'
plt.savefig(OUT, dpi=140, bbox_inches='tight')
from IPython.display import display, Image as _Img
display(_Img(filename=OUT))
print(f'Saved -> {OUT}')

### Side-by-side detection comparison — do the boxes match?

In [ ]:
from detector import Detector, CATEGORY_OF, display_label, MISSION_CLASSES
from mapper   import CATEGORY_COLORS

def run_onnx_on_image(model_path, img, conf=0.30):
    """Run an ONNX model on a BGR image, return list of detection dicts."""
    det = Detector.__new__(Detector)
    det.model_path = model_path
    det.conf = conf
    det.imgsz = 416
    det.available = True
    det._backend = 'onnx'
    det._cv2 = cv2
    det._np  = np
    # Load ONNX session
    import onnxruntime as ort, ast
    from detector import _COCO80_NAMES
    sess = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    det._session = sess
    det._ort_input_name = sess.get_inputs()[0].name
    meta = sess.get_modelmeta().custom_metadata_map
    names_str = meta.get('names', '')
    det._ort_names = ast.literal_eval(names_str) if names_str else _COCO80_NAMES
    return det._infer_onnx(img)

test_img_name = list(images.keys())[0]   # workspace.jpg
test_img      = images[test_img_name]

fp32_dets = run_onnx_on_image(FP32_PATH, test_img)
int8_dets = run_onnx_on_image(INT8_PATH, test_img)

print(f'FP32 detections on {test_img_name}: {len(fp32_dets)}')
for d in fp32_dets:
    print(f'  {d["category"]:<12} {d["label"]:<10} conf={d["confidence"]:.0%}  cx={d["bbox_cx_frac"]:.2f}')
print(f'INT8 detections on {test_img_name}: {len(int8_dets)}')
for d in int8_dets:
    print(f'  {d["category"]:<12} {d["label"]:<10} conf={d["confidence"]:.0%}  cx={d["bbox_cx_frac"]:.2f}')

In [ ]:
def draw_dets(ax, img, dets, title):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for d in dets:
        # Reconstruct approx x1,y1 from cx_frac (illustrative — no full bbox from ONNX path)
        col  = CATEGORY_COLORS.get(d['category'], '#c0c0c0')
        cx   = d['bbox_cx_frac'] * img.shape[1]
        cy   = img.shape[0] * 0.4
        size = 60
        pts  = np.array([[cx, cy-size],[cx+size,cy],[cx,cy+size],[cx-size,cy]], np.int32)
        poly = mpatches.Polygon(pts, closed=True, linewidth=2.5,
                                edgecolor=col, facecolor=col+'30')
        ax.add_patch(poly)
        ax.text(cx, cy-size-8, f"{d['label']} {d['confidence']:.0%}",
                ha='center', color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=col, alpha=0.85, pad=1.5, edgecolor='none'))
    ax.set_title(title, fontsize=11)
    ax.axis('off')

# If we have full bbox from YOLO_AVAILABLE, use those for FP32
fp32_src = all_results.get(test_img_name, [])

fig, (ax_fp32, ax_int8) = plt.subplots(1, 2, figsize=(14, 6))

# Draw FP32 using Ultralytics bboxes if available, else ONNX
if YOLO_AVAILABLE and fp32_src and 'bbox' in fp32_src[0]:
    ax_fp32.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    seen = set()
    for d in fp32_src:
        col = CATEGORY_COLORS.get(CATEGORY_OF.get(d['coco'], ''), '#ccc')
        x1,y1,x2,y2 = d['bbox']
        rect = mpatches.FancyBboxPatch((x1,y1),x2-x1,y2-y1,
                   boxstyle='round,pad=2',linewidth=2.5,edgecolor=col,facecolor='none')
        ax_fp32.add_patch(rect)
        ax_fp32.text(x1+3, max(y1-6,8), f"{display_label(d['coco'])} {d['conf']:.0%}",
               color='white',fontsize=8,fontweight='bold',
               bbox=dict(facecolor=col,alpha=0.85,pad=1.5,edgecolor='none'))
    ax_fp32.set_title(f'FP32 Ultralytics ({fp32_mb:.1f} MB)', fontsize=11)
    ax_fp32.axis('off')
else:
    draw_dets(ax_fp32, test_img, fp32_dets, f'FP32 ONNX ({fp32_mb:.1f} MB)')

draw_dets(ax_int8, test_img, int8_dets, f'INT8 ONNX ({int8_mb:.1f} MB) — edge model')

plt.suptitle('Detection quality: FP32 vs INT8  (same objects, smaller model)',
             fontsize=13)
plt.tight_layout()
OUT = '/tmp/darkmap_fp32_vs_int8.png'
plt.savefig(OUT, dpi=140, bbox_inches='tight')
from IPython.display import display, Image as _Img
display(_Img(filename=OUT))
print(f'Saved -> {OUT}')

### Deploy to the UNO Q

```bash
# Copy the INT8 model to the board (only ~3 MB)
scp linux/models/yolov8n_int8.onnx arduino@uno-q:~/darkmap/

# Install only onnxruntime on the board — no PyTorch, no Ultralytics
pip install onnxruntime opencv-python

# Run with the INT8 ONNX model
python3 main.py --source serial --model ~/darkmap/yolov8n_int8.onnx
```

The `--model` flag auto-selects the ONNX Runtime backend in `detector.py`
when the path ends with `.onnx`.